# 📈 SBER 5-Minute Intraday Research Pipeline

**Goal:** feature engineering on raw Finam OHLCV candles → XGBoost classifier → probability calibration → feature importance.

**Feature source:** [`stock-alpha-lab/features/candle_features.py`](https://github.com/jekamalyshev/stock-alpha-lab/blob/main/features/candle_features.py)

**Target variable:** `target_is_green_next = 1` if next 5m candle closes above its open, else 0.

**Supervised expansion:** `series_to_supervised(n_in=3)` — adds lags t-1, t-2, t-3 for every feature column.

**Column count formula after lag expansion:**
```
output_cols = n_feature_cols × (n_in + 1) + 1 (target)
```

**Important:** all features use only current-bar-close or past-bar data. No look-ahead bias.

## Cell 1 — Imports & Display Settings

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (15, 5)

import sklearn
from packaging.version import Version
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss,
    brier_score_loss, classification_report, confusion_matrix,
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

# ── stock-alpha-lab feature module ──────────────────────────────────────────
# Добавляем путь к stock-alpha-lab, если он лежит рядом:
#   ../stock-alpha-lab/features/candle_features.py
# Либо установи как пакет: pip install -e ../stock-alpha-lab
_ALPHA_LAB_PATH = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..', 'stock-alpha-lab'))
if _ALPHA_LAB_PATH not in sys.path:
    sys.path.insert(0, _ALPHA_LAB_PATH)

from features.candle_features import build_sber_5m_candle_features, series_to_supervised

_SKLEARN_VERSION = Version(sklearn.__version__)
_CALIB_ESTIMATOR_KWARG = 'estimator' if _SKLEARN_VERSION >= Version('1.2') else 'base_estimator'

print('Python :', sys.version)
print('pandas :', pd.__version__)
print('numpy  :', np.__version__)
print('sklearn:', sklearn.__version__)
print('xgboost:', __import__('xgboost').__version__)
print(f'CalibratedClassifierCV kwarg: "{_CALIB_ESTIMATOR_KWARG}"')
print('All imports OK ✓')

## Cell 2 — Calendar Features

Все остальные признаки (candle geometry, TA, returns, rolling) строятся через `build_sber_5m_candle_features`.
Здесь добавляем только intraday-calendar признаки, которых нет в candle_features.py.

In [ ]:
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add intraday calendar features based on 'datetime' column.

    Requires 'datetime' column — produced by build_sber_5m_candle_features
    when DATE and TIME columns are present.
    """
    if 'datetime' not in df.columns:
        raise ValueError("Column 'datetime' not found. Run build_sber_5m_candle_features first.")
    out = df.copy(deep=True)
    dt = out['datetime']
    out['hour']                      = dt.dt.hour
    out['minute']                    = dt.dt.minute
    out['dayofweek']                 = dt.dt.dayofweek
    out['hour_sin']                  = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos']                  = np.cos(2 * np.pi * out['hour'] / 24)
    out['dow_sin']                   = np.sin(2 * np.pi * out['dayofweek'] / 5)
    out['dow_cos']                   = np.cos(2 * np.pi * out['dayofweek'] / 5)
    out['minutes_from_session_open'] = out['hour'] * 60 + out['minute'] - 600
    out['is_opening_30m']            = (out['minutes_from_session_open'] < 30).astype('int8')
    out['is_first_hour']             = (out['minutes_from_session_open'] < 60).astype('int8')
    out['is_closing_30m']            = (out['minutes_from_session_open'] >= 390).astype('int8')
    out['date_only']                 = dt.dt.date
    out['bar_in_day']                = out.groupby('date_only').cumcount()
    out['is_first_bar_of_day']       = (out['bar_in_day'] == 0).astype('int8')
    out = out.drop(columns=['date_only'])
    return out


print('add_calendar_features defined ✓')

## Cell 3 — Target & Dataset Builder

In [ ]:
def add_classification_target(df: pd.DataFrame) -> pd.DataFrame:
    """Append binary target: 1 if next bar closes above its open, else 0.

    NOTE: drops the last row (no next bar available).
    This target is INDEPENDENT of target_ret_4 produced by
    build_sber_5m_candle_features — both can coexist.
    """
    out = df.copy(deep=True)
    out['next_open']  = out['OPEN'].shift(-1)
    out['next_close'] = out['CLOSE'].shift(-1)
    out['target_is_green_next'] = (out['next_close'] > out['next_open']).astype('int8')
    out = out.drop(columns=['next_open', 'next_close'])
    out = out.iloc[:-1].reset_index(drop=True)
    return out


_LEAKAGE_COLS = [
    'datetime', 'DATE', 'TIME', 'TICKER', 'PER',
    'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOL',
]


def build_X_y_for_model(
    feature_df: pd.DataFrame,
    n_in: int = 3,
    target_col: str = 'target_is_green_next',
) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    """Build supervised dataset with series_to_supervised(n_in) lag expansion.

    Parameters
    ----------
    feature_df : output of the full pipeline (build_sber_5m_candle_features
                 + add_calendar_features + add_classification_target).
    n_in       : lag window. With n_in=3 each sample sees t, t-1, t-2, t-3.
    target_col : binary classification target column.

    Returns
    -------
    X : pd.DataFrame — supervised feature matrix
    y : pd.Series    — target vector
    leakage_cols : list[str] — columns excluded as leakage / raw OHLCV

    Column count formula
    --------------------
    n_feature_cols × (n_in + 1) + 1 (target)
    e.g. for n_feature_cols=120, n_in=3:  120 × 4 + 1 = 481
    """
    all_target_cols = [c for c in feature_df.columns if c.startswith('target_')]
    leakage_cols    = list(set(_LEAKAGE_COLS + all_target_cols))
    feature_cols    = [c for c in feature_df.columns if c not in leakage_cols]

    X_base = feature_df[feature_cols].copy()
    y_full = feature_df[target_col].copy()

    n_raw = X_base.shape[1]

    valid_idx = X_base.dropna().index
    X_clean   = X_base.loc[valid_idx].reset_index(drop=True)
    y_clean   = y_full.loc[valid_idx].reset_index(drop=True)

    # series_to_supervised: creates col(t-n_in)…col(t-1), col(t)
    # We pass a temporary DataFrame where target_col is added as last column
    # so it's carried through — but here we manage target separately.
    cols, names = [], []
    for i in range(n_in, 0, -1):
        cols.append(X_clean.shift(i))
        names += [f'{c}(t-{i})' for c in X_clean.columns]
    cols.append(X_clean)
    names += [f'{c}(t)' for c in X_clean.columns]

    X_supervised = pd.concat(cols, axis=1)
    X_supervised.columns = names
    X_supervised = X_supervised.dropna().reset_index(drop=True)

    # align y: first n_in rows are lost after lagging
    y_aligned = y_clean.iloc[n_in:].reset_index(drop=True)
    assert len(X_supervised) == len(y_aligned), (
        f'Length mismatch: X={len(X_supervised)}, y={len(y_aligned)}'
    )

    n_sup = X_supervised.shape[1]
    expected = n_raw * (n_in + 1)
    check = '✓ OK' if n_sup == expected else '✗ MISMATCH'

    sep = '─' * 60
    print(sep)
    print(f'  Feature count BEFORE series_to_supervised : {n_raw}')
    print(f'  Lag window (n_in)                          : {n_in}')
    print(f'  Formula: {n_raw} × (n_in={n_in}+1) = {expected}')
    print(f'  Feature count AFTER  series_to_supervised : {n_sup}  {check}')
    print(f'  Dataset shape (rows × cols)                : {X_supervised.shape}')
    print(sep)

    return X_supervised, y_aligned, leakage_cols


print('Target & dataset builder functions defined ✓')

## Cell 4 — Load Data

In [ ]:
DATA_PATH = './data/Сбербанк/year_result.csv'
frame = pd.read_csv(DATA_PATH, header=0, sep=';')
frame.columns = [c.strip('<>').strip() for c in frame.columns]
print(f'Загружено строк : {len(frame):,}')
print(f'Колонки         : {frame.columns.tolist()}')
frame.head(3)

## Cell 5 — Build Features

`build_sber_5m_candle_features` строит:
- candle geometry (~16 cols)
- returns / momentum (~10 cols)
- rolling context ×3 windows (~33 cols)
- volume interaction (~4 cols)
- TA indicators via `ta` library (~80 cols)
- `target_ret_4` (forward return, label only)

Затем добавляем calendar features и classification target.

In [ ]:
# ── Stage 1: candle + TA features from stock-alpha-lab ──────────────────
# apply_supervised=False — lag expansion будет сделан в Cell 7
# verbose=True — печатает отчёт по количеству признаков
df_feat = build_sber_5m_candle_features(
    frame,
    windows=(6, 12, 24),
    ta_window=14,
    shift=4,
    add_target=True,         # target_ret_4 — continuous label (не используем как feature)
    apply_supervised=False,  # lag expansion — отдельно в Cell 7
    verbose=True,
)

# ── Stage 2: calendar features ─────────────────────────────────────────
df_feat = add_calendar_features(df_feat)

# ── Stage 3: binary classification target ──────────────────────────────
df_feat = add_classification_target(df_feat)

# ── Report ──────────────────────────────────────────────────────────────
leakage_for_report = list(set(_LEAKAGE_COLS + [c for c in df_feat.columns if c.startswith('target_')]))
feature_cols_report = [c for c in df_feat.columns if c not in leakage_for_report]

print(f'\nTotal feature columns (before lag expansion) : {len(feature_cols_report)}')
print(f'Total rows after pipeline                    : {df_feat.shape[0]:,}')
print(f'\nClass balance (target_is_green_next):')
print(df_feat['target_is_green_next'].value_counts(normalize=True)
      .rename({0: 'red (0)', 1: 'green (1)'}).round(3))
df_feat.head(3)

## Cell 6 — EDA

In [ ]:
eda_cols = ['ret_1', 'body_to_range', 'close_pos_in_range', 'range_pct_close',
            'vol_ratio_6', 'rsi', 'close_vs_ma_12', 'adx', 'cmf', 'mfi']
eda_cols = [c for c in eda_cols if c in df_feat.columns]
fig, axes = plt.subplots(1, len(eda_cols), figsize=(22, 4))
for ax, col in zip(axes, eda_cols):
    df_feat[col].dropna().hist(bins=60, ax=ax, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(col, fontsize=9)
plt.suptitle('Feature distributions (raw, before lag expansion)', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
top_feat = ['ret_1', 'ret_6', 'ret_12', 'body_to_range', 'close_pos_in_range',
            'vol_ratio_6', 'rsi', 'close_vs_ma_12', 'range_zscore_12',
            'adx', 'cmf', 'mfi', 'close_vs_vwap',
            'is_opening_30m', 'direction', 'target_is_green_next']
top_feat = [c for c in top_feat if c in df_feat.columns]
plt.figure(figsize=(14, 11))
sns.heatmap(df_feat[top_feat].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix — Key Features + Target')
plt.tight_layout(); plt.show()

## Cell 7 — Build Supervised Dataset & Time Split

`series_to_supervised(n_in=3)` создаёт лаги t-1, t-2, t-3 для каждого признака.

**Формула:** `n_feature_cols × (n_in + 1)` — умножаем на 4 (текущий бар + 3 лага).

In [ ]:
X, y, leakage_cols = build_X_y_for_model(df_feat, n_in=3)

n = len(X)
n_train = int(n * 0.70)
n_valid = int(n * 0.15)
X_train, y_train = X.iloc[:n_train],                y.iloc[:n_train]
X_valid, y_valid = X.iloc[n_train:n_train+n_valid], y.iloc[n_train:n_train+n_valid]
X_calib, y_calib = X.iloc[n_train+n_valid:],        y.iloc[n_train+n_valid:]

print()
for name, Xs, ys in [('Train', X_train, y_train), ('Valid', X_valid, y_valid), ('Calib', X_calib, y_calib)]:
    bal = ys.value_counts(normalize=True).round(3).to_dict()
    print(f'{name:6s}: {len(Xs):6,} rows | class balance: {bal}')

## Cell 8 — XGBoost Classifier

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_va, y_va, X_ca, y_ca, model_name='Model'):
    results = {}
    for split_name, Xs, ys in [('Train', X_tr, y_tr), ('Valid', X_va, y_va), ('Calib', X_ca, y_ca)]:
        preds  = model.predict(Xs)
        probas = model.predict_proba(Xs)[:, 1]
        acc   = accuracy_score(ys, preds)
        auc   = roc_auc_score(ys, probas)
        ll    = log_loss(ys, probas)
        brier = brier_score_loss(ys, probas)
        results[split_name] = {'accuracy': acc, 'roc_auc': auc, 'log_loss': ll, 'brier': brier}
        print(f'[{model_name}] {split_name:6s} | Acc={acc:.4f} AUC={auc:.4f} LogLoss={ll:.4f} Brier={brier:.4f}')
    return results


_neg = int((y_train == 0).sum())
_pos = int((y_train == 1).sum())
_scale_pos_weight = _neg / _pos if _pos > 0 else 1.0
print(f'Class balance → neg={_neg:,}, pos={_pos:,}, scale_pos_weight={_scale_pos_weight:.4f}')

baseline_model = XGBClassifier(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.5,
    min_child_weight=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=_scale_pos_weight,
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

baseline_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
print(f'Best iteration: {baseline_model.best_iteration}')

print('\n=== Baseline: XGBoost ===')
baseline_results = evaluate_model(baseline_model, X_train, y_train, X_valid, y_valid, X_calib, y_calib, 'XGB')

In [ ]:
print('\n=== Classification Report (Valid) ===')
print(classification_report(y_valid, baseline_model.predict(X_valid), target_names=['red (0)', 'green (1)']))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_valid, baseline_model.predict(X_valid))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Red', 'Pred Green'],
            yticklabels=['Actual Red', 'Actual Green'])
plt.title('Confusion Matrix — XGBoost (Valid)')
plt.tight_layout(); plt.show()

## Cell 9 — Probability Calibration (Platt Scaling)

In [ ]:
calibrated_model = CalibratedClassifierCV(
    **{_CALIB_ESTIMATOR_KWARG: baseline_model},
    method='sigmoid',
    cv='prefit',
)
calibrated_model.fit(X_calib, y_calib)

print('=== Calibrated XGBoost ===')
calib_results = evaluate_model(calibrated_model, X_train, y_train, X_valid, y_valid, X_calib, y_calib, 'XGB+Calib')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for model, label, color in [
    (baseline_model, 'Baseline XGB', 'steelblue'),
    (calibrated_model, 'XGB + Platt', 'tomato'),
]:
    prob_true, prob_pred = calibration_curve(y_valid, model.predict_proba(X_valid)[:, 1], n_bins=10)
    ax.plot(prob_pred, prob_true, marker='o', label=label, color=color)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curve (Valid)'); ax.legend()
axes[1].hist(baseline_model.predict_proba(X_valid)[:, 1], bins=40, alpha=0.6, label='Baseline XGB', color='steelblue')
axes[1].hist(calibrated_model.predict_proba(X_valid)[:, 1], bins=40, alpha=0.6, label='XGB + Platt', color='tomato')
axes[1].set_xlabel('Predicted probability'); axes[1].set_title('Probability Distribution (Valid)'); axes[1].legend()
plt.tight_layout(); plt.show()

## Cell 10 — Metrics Summary Table

In [ ]:
rows = []
for split in ['Train', 'Valid', 'Calib']:
    r_base  = baseline_results[split]
    r_calib = calib_results[split]
    rows.append({
        'Split':        split,
        'XGB Accuracy': round(r_base['accuracy'],  4),
        'XGB AUC':      round(r_base['roc_auc'],   4),
        'XGB LogLoss':  round(r_base['log_loss'],  4),
        'XGB Brier':    round(r_base['brier'],     4),
        'Cal Accuracy': round(r_calib['accuracy'], 4),
        'Cal AUC':      round(r_calib['roc_auc'],  4),
        'Cal LogLoss':  round(r_calib['log_loss'], 4),
        'Cal Brier':    round(r_calib['brier'],    4),
    })

metrics_df = pd.DataFrame(rows).set_index('Split')
print('\n=== Metrics Summary: XGBoost vs Calibrated ===')
display(
    metrics_df.style
    .format(precision=4)
    .highlight_min(subset=['XGB LogLoss', 'XGB Brier', 'Cal LogLoss', 'Cal Brier'], color='#d4edda')
    .highlight_max(subset=['XGB Accuracy', 'XGB AUC', 'Cal Accuracy', 'Cal AUC'], color='#d4edda')
    .set_caption('Green = best value per column')
)

## Cell 11 — Feature Importance

In [ ]:
TOP_N = 40

importance_gain = (
    pd.Series(baseline_model.get_booster().get_score(importance_type='gain'))
    .sort_values(ascending=False)
    .head(TOP_N)
)

perm_result = permutation_importance(
    baseline_model, X_valid, y_valid,
    n_repeats=10, scoring='roc_auc', random_state=42, n_jobs=-1,
)
perm_df = (
    pd.DataFrame({'feature': X_valid.columns,
                  'importance': perm_result.importances_mean,
                  'std': perm_result.importances_std})
    .sort_values('importance', ascending=False)
    .head(TOP_N)
)

fig, axes = plt.subplots(1, 2, figsize=(24, 14))
importance_gain.sort_values().plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title(f'XGBoost Feature Importance — Gain (top {TOP_N})', fontsize=13)
axes[0].set_xlabel('Mean Gain')
axes[0].axvline(importance_gain.mean(), color='tomato', linestyle='--', lw=1.5, label='Mean')
axes[0].legend()
perm_plot = perm_df.sort_values('importance')
axes[1].barh(perm_plot['feature'], perm_plot['importance'], xerr=perm_plot['std'],
             color='seagreen', edgecolor='none', alpha=0.85, capsize=3)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title(f'Permutation Importance (AUC, Valid, top {TOP_N})', fontsize=13)
axes[1].set_xlabel('Mean AUC decrease when feature permuted')
plt.suptitle('XGBoost Feature Importance Analysis', fontsize=15, y=1.01)
plt.tight_layout(); plt.show()

print('\n=== Top-30 Features by Permutation Importance (Valid AUC) ===')
display(
    perm_df.head(30)
    .assign(importance=lambda d: d['importance'].round(6),
            std=lambda d: d['std'].round(6))
    .reset_index(drop=True)
    .style.bar(subset=['importance'], color='#5fba7d', vmin=0)
    .format({'importance': '{:.6f}', 'std': '±{:.6f}'})
    .set_caption('Permutation Importance: mean AUC drop on Valid set (n_repeats=10)')
)

zero_perm = perm_df[perm_df['importance'] <= 0]['feature'].tolist()
if zero_perm:
    print(f'\n⚠️  Признаки с importance ≤ 0 (кандидаты на удаление): {len(zero_perm)}')
    print(zero_perm)
else:
    print('\n✓ Все признаки имеют положительную permutation importance.')

## Next Steps

- [ ] Walk-forward validation (expanding window)
- [ ] Убрать признаки с permutation importance ≤ 0 → переобучить
- [ ] Hyperparameter tuning (Optuna / Bayesian search)
- [ ] Сменить таргет: Triple Barrier Label / ret_Nm > fee_threshold
- [ ] Симуляция транзакционных издержек
- [ ] Regime detection (rolling volatility / HMM)
- [ ] Meta-labeling для фильтрации сигналов